# Report your own property

*Data Analysis in Python · Topic 12.6*

Your team wants the last 28 days by channel group from your own GA4 property. Set up a service account for the property, and pull the report with it.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Set up the service account as in the video: turn on the Google Analytics Data API in a Google Cloud project, create a service account with a JSON key, add its email address as a Viewer in your GA4 property, and add the key to Colab Secrets as `GA4_SERVICE_ACCOUNT`, with **Notebook access** on.
2. In the parameters cell, set `PROPERTY_ID` to your property's ID, from **Admin → Property details** in GA4. Run the cell that installs `google-analytics-data`, then the parameters cell.
3. In the first cell marked ✏️, create `client`: the GA4 client, logged in with the key in `GA4_SERVICE_ACCOUNT`.
4. In the second cell marked ✏️, create `response`: a report for `START_DATE` to `END_DATE`, with the dimension `sessionDefaultChannelGroup` and the metrics `sessions` and `totalUsers`, in this order.
5. In the last cell marked ✏️, create the `channels` table: one row per channel group, with the API names as column names, and `sessions` and `totalUsers` as whole numbers.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then compare the sessions with GA4's **Traffic acquisition** report for the last 28 days.

**Expected output**: The `channels` table has one row per channel group in your property, with its sessions and total users as numbers. Its sessions match GA4's **Traffic acquisition** report for the last 28 days.

In [ ]:
%pip install google-analytics-data

## Parameters

In [ ]:
import json

import pandas as pd
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import DateRange, Dimension, Metric, RunReportRequest
from google.colab import userdata

PROPERTY_ID = ""  # Your property's ID, from Admin → Property details in GA4
START_DATE, END_DATE = "28daysAgo", "yesterday"  # The last 28 days, counted back from today

## The client

In [ ]:
key = json.loads(userdata.get("GA4_SERVICE_ACCOUNT"))
client = BetaAnalyticsDataClient.from_service_account_info(key)

## The report

In [ ]:
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="sessionDefaultChannelGroup")],
    metrics=[Metric(name="sessions"), Metric(name="totalUsers")],
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
)
response = client.run_report(request)
response.row_count

## The table

In [ ]:
names = [h.name for h in response.dimension_headers] + [h.name for h in response.metric_headers]
rows = [[v.value for v in row.dimension_values] + [v.value for v in row.metric_values] for row in response.rows]
channels = pd.DataFrame(rows, columns=names)
channels[["sessions", "totalUsers"]] = channels[["sessions", "totalUsers"]].astype(int)
channels

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source and not source.lstrip().startswith("%pip")] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "SecretNotFound" in message:
        results.append((None, "A cell stopped because Colab Secrets has no GA4_SERVICE_ACCOUNT. Check the secret's name, then choose Runtime → Restart session and run all."))
    elif "NotebookAccess" in message:
        results.append((None, "A cell stopped because GA4_SERVICE_ACCOUNT doesn't have Notebook access. Switch it on in Colab Secrets, then choose Runtime → Restart session and run all."))
    elif "SERVICE_DISABLED" in message or "has not been used in project" in message:
        results.append((None, "The Google Analytics Data API isn't turned on in the service account's Google Cloud project. Turn it on under APIs & Services → Library, wait a minute, then choose Runtime → Restart session and run all."))
    elif "PermissionDenied" in message or "403" in message:
        results.append((None, "GA4 answered 403: the service account can't read this property. Check that PROPERTY_ID is your property's ID, and that the service account's email address is a Viewer under Admin → Property access management. New access can take a few minutes."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    property_id = found.get("PROPERTY_ID")
    if not isinstance(property_id, str) or not property_id.isdigit():
        results.append((False, f"PROPERTY_ID should be your property's ID, a number in quotes, from Admin → Property details in GA4. It's {property_id!r}."))

    client = found.get("client")
    if type(client).__name__ != "BetaAnalyticsDataClient":
        results.append((False, "client should be the GA4 client, logged in with your service account: BetaAnalyticsDataClient.from_service_account_info(key), where key is json.loads(userdata.get(\"GA4_SERVICE_ACCOUNT\"))."))

    response = found.get("response")
    if response is None or not hasattr(response, "rows"):
        results.append((False, "response should be the answer of client.run_report(request)."))
    else:
        headers = [h.name for h in response.dimension_headers] + [h.name for h in response.metric_headers]
        if headers != ["sessionDefaultChannelGroup", "sessions", "totalUsers"]:
            results.append((False, f"The report should have the dimension sessionDefaultChannelGroup and the metrics sessions and totalUsers, in this order. It has {headers}."))
        elif response.row_count == 0:
            results.append((False, f"The report is empty: property {property_id} has no sessions from {found.get('START_DATE')} to {found.get('END_DATE')}. Check that PROPERTY_ID is the property you added the service account to."))
        else:
            results.append((True, f"A report of {response.row_count} channel groups from property {property_id}."))

    channels = found.get("channels")
    columns = ["sessionDefaultChannelGroup", "sessions", "totalUsers"]
    if not isinstance(channels, pd.DataFrame) or [column for column in columns if column not in channels.columns]:
        results.append((False, f"channels should be a DataFrame with the columns {', '.join(columns)}."))
    elif response is not None and hasattr(response, "rows") and len(channels) != len(response.rows):
        results.append((False, f"The channels table should have one row per row of the report, {len(response.rows)}. It has {len(channels)}."))
    elif not all(pd.api.types.is_numeric_dtype(channels[column]) for column in columns[1:]):
        results.append((False, "The metrics are still text: GA4 sends every value as a string. Convert sessions and totalUsers with astype(int)."))
    elif response is not None and hasattr(response, "rows") and int(channels["sessions"].sum()) != sum(int(row.metric_values[0].value) for row in response.rows):
        results.append((False, "The channels table's sessions don't add up to the report's. Check that the metrics are in the order you asked for them."))
    else:
        results.append((True, f"{len(channels)} channel groups and {int(channels['sessions'].sum()):,} sessions, the same as the report. Compare them with GA4's Traffic acquisition report for the same dates."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
key = json.loads(userdata.get("GA4_SERVICE_ACCOUNT"))
client = BetaAnalyticsDataClient.from_service_account_info(key)
```

```python
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="sessionDefaultChannelGroup")],
    metrics=[Metric(name="sessions"), Metric(name="totalUsers")],
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
)
response = client.run_report(request)
response.row_count
```

```python
names = [h.name for h in response.dimension_headers] + [h.name for h in response.metric_headers]
rows = [[v.value for v in row.dimension_values] + [v.value for v in row.metric_values] for row in response.rows]
channels = pd.DataFrame(rows, columns=names)
channels[["sessions", "totalUsers"]] = channels[["sessions", "totalUsers"]].astype(int)
channels
```

</details>